# Theorem 2 Task B: environment modes workflow

This notebook keeps the arithmetic problem family fixed and changes only the visible public environment:

- `artifact_only`
- `worked_trace`
- `failed_then_repair`

It then evaluates the same learner class in two deployment modes:

- direct-answer mode
- process mode


In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

def env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == "Drift_and_selection":
            return p
        if (p / "GitHub").exists() and (p / "Nat_Paper").exists():
            return p
    return start

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "GitHub" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".mplconfig"))

from drift_selection.theorem2_environment_modes import (
    STYLES,
    EnvironmentModesConfig,
    ModelConfig,
    TrainConfig,
    build_environment_datasets,
    find_task_a_summary_csv,
    pretty_print_examples,
    prompt_budget_summary,
    run_environment_modes_workflow,
)


Matplotlib is building the font cache; this may take a moment.


## Run order

1. Preview only: `RUN_BUILD_PREVIEWS = True`, `RUN_PIPELINE = False`, `BUILD_APPENDIX_ASSETS = False`
2. Full run: set `RUN_PIPELINE = True`
3. Appendix assets: also set `BUILD_APPENDIX_ASSETS = True` and point `TASK_A_SUMMARY_CSV` at the existing Task A summary if needed


In [2]:
RUN_BUILD_PREVIEWS = env_flag("RUN_BUILD_PREVIEWS", True)
RUN_PIPELINE = env_flag("RUN_PIPELINE", False)
BUILD_APPENDIX_ASSETS = env_flag("BUILD_APPENDIX_ASSETS", False)

RUN_NAME = os.getenv("RUN_NAME", "theorem2_taskB_environment_modes_v2")
TASK_A_SUMMARY_CSV = os.getenv("TASK_A_SUMMARY_CSV", "")

ENV_CFG = EnvironmentModesConfig(
    train_min_digits=2,
    train_max_digits=3,
    id_test_min_digits=2,
    id_test_max_digits=3,
    ood_test_min_digits=4,
    ood_test_max_digits=4,
    n_train=9000,
    n_val=1200,
    n_id_test=2000,
    n_ood_test=1600,
    equalize_examples_to_longest=True,
    include_both_prompt_modes_in_train=True,
)

MODEL_CFG = ModelConfig(
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    dropout=0.1,
    max_seq_len=256,
)

TRAIN_CFG = TrainConfig(
    batch_size=64,
    learning_rate=3e-4,
    weight_decay=0.01,
    epochs=8,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_NAME:", RUN_NAME)
print("RUN_BUILD_PREVIEWS:", RUN_BUILD_PREVIEWS)
print("RUN_PIPELINE:", RUN_PIPELINE)
print("BUILD_APPENDIX_ASSETS:", BUILD_APPENDIX_ASSETS)
print("TASK_A_SUMMARY_CSV:", TASK_A_SUMMARY_CSV or "<auto-detect>")


PROJECT_ROOT: /Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection
RUN_NAME: theorem2_taskB_environment_modes_v2
RUN_BUILD_PREVIEWS: True
RUN_PIPELINE: False
BUILD_APPENDIX_ASSETS: False
TASK_A_SUMMARY_CSV: <auto-detect>


In [3]:
if RUN_BUILD_PREVIEWS:
    preview_cfg = EnvironmentModesConfig(
        train_min_digits=ENV_CFG.train_min_digits,
        train_max_digits=ENV_CFG.train_max_digits,
        id_test_min_digits=ENV_CFG.id_test_min_digits,
        id_test_max_digits=ENV_CFG.id_test_max_digits,
        ood_test_min_digits=ENV_CFG.ood_test_min_digits,
        ood_test_max_digits=ENV_CFG.ood_test_max_digits,
        n_train=24,
        n_val=8,
        n_id_test=8,
        n_ood_test=8,
        equalize_examples_to_longest=ENV_CFG.equalize_examples_to_longest,
        include_both_prompt_modes_in_train=ENV_CFG.include_both_prompt_modes_in_train,
    )
    preview_datasets, preview_manifest = build_environment_datasets(preview_cfg, styles=list(STYLES))
    print("Train prompt modes:", preview_manifest["train_prompt_modes"])
    for style in STYLES:
        print(f"\nSTYLE: {style}")
        print(pretty_print_examples(preview_datasets[style]["train"], n=2))
    display(prompt_budget_summary(preview_datasets))


Train prompt modes: ['DIRECT', 'WORK']

STYLE: artifact_only
PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE DIRECT
TARGET: ANS 1 3 6 4

PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE WORK
TARGET: ANS 1 3 6 4


STYLE: worked_trace
PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE DIRECT
TARGET: WORK COL 0 5 9 SUM 1 4 W 4 C 1 COL 1 2 3 CIN 1 SUM 6 W 6 C 0 COL 2 7 6 SUM 1 3 W 3 C 1 ANS 1 3 6 4

PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE WORK
TARGET: WORK COL 0 5 9 SUM 1 4 W 4 C 1 COL 1 2 3 CIN 1 SUM 6 W 6 C 0 COL 2 7 6 SUM 1 3 W 3 C 1 ANS 1 3 6 4


STYLE: failed_then_repair
PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE DIRECT
TARGET: TRY NOCARRY 3 5 4 FAIL REPAIR COL 0 5 9 SUM 1 4 W 4 C 1 COL 1 2 3 CIN 1 SUM 6 W 6 C 0 COL 2 7 6 SUM 1 3 W 3 C 1 ANS 1 3 6 4

PROMPT: TASK TASKB_ENV_ADD A 7 2 5 SEP B 6 3 9 MODE WORK
TARGET: TRY NOCARRY 3 5 4 FAIL REPAIR COL 0 5 9 SUM 1 4 W 4 C 1 COL 1 2 3 CIN 1 SUM 6 W 6 C 0 COL 2 7 6 SUM 1 3 W 3 C 1 ANS 1 3 6 4



,style,split,n_examples,avg_prompt_len,avg_target_len
0,artifact_only,train,48,12.250,4.291667
1,artifact_only,val,16,12.125,4.125000
2,artifact_only,id_test_direct,8,12.375,4.375000
3,artifact_only,id_test_process,8,12.375,4.375000
4,artifact_only,ood_test_direct,8,15.000,5.625000
5,artifact_only,ood_test_process,8,15.000,5.625000
6,worked_trace,train,48,12.250,37.708333
7,worked_trace,val,16,12.125,37.125000
8,worked_trace,id_test_direct,8,12.375,38.000000
9,worked_trace,id_test_process,8,12.375,38.000000


In [4]:
task_b_result = None

if RUN_PIPELINE:
    task_a_path = Path(TASK_A_SUMMARY_CSV) if TASK_A_SUMMARY_CSV else find_task_a_summary_csv(PROJECT_ROOT)
    task_b_result = run_environment_modes_workflow(
        env_cfg=ENV_CFG,
        model_cfg=MODEL_CFG,
        train_cfg=TRAIN_CFG,
        run_name=RUN_NAME,
        project_root=PROJECT_ROOT,
        task_a_summary_csv=task_a_path,
        build_appendix_assets_flag=BUILD_APPENDIX_ASSETS,
    )
    print("Task B run root:", task_b_result["run_root"])
    print("Task A summary used:", task_b_result["task_a_summary_csv"])
    display(task_b_result["direct_summary"])
    display(task_b_result["process_summary"])
    print("Direct summary:", task_b_result["evaluation_dir"] / "summary_direct_mode.csv")
    print("Process summary:", task_b_result["evaluation_dir"] / "summary_process_mode.csv")
    print("Direct metrics figure:", task_b_result["figure_paths"]["direct_metrics"])
    print("Process metrics figure:", task_b_result["figure_paths"]["process_metrics"])
    print("Direct contamination figure:", task_b_result["figure_paths"]["direct_contamination"])
    print("Appendix CSV:", task_b_result["appendix_paths"]["combined_csv"])
    print("Appendix table:", task_b_result["appendix_paths"]["table_tex"])


In [5]:
run_root = PROJECT_ROOT / "GitHub" / "data" / "outputs" / "theorem2_process_learning" / "taskB_environment_modes_v2" / RUN_NAME
direct_csv = run_root / "evaluation" / "summary_direct_mode.csv"
process_csv = run_root / "evaluation" / "summary_process_mode.csv"
if direct_csv.exists():
    display(pd.read_csv(direct_csv))
if process_csv.exists():
    display(pd.read_csv(process_csv))
